# 01 — Timing Benchmark

Determine the appropriate SIPNET simulation time horizon for MCMC.
**Decision rule:** choose the longest horizon where mean run time < 100 ms/run.

> **Result recorded after execution:** see the comment at the top of the benchmark cell.


In [ ]:
# RESULT: benchmark_climate.clim was written with the chosen horizon.
# See the "Chosen horizon" print output below for the selected period.
import time
import tempfile
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from pysipnet import SIPNETModel, SIPNETRunner, ClimateDrivers
from pysipnet.runner import ClimateStaging
from pysipnet.parameters import (
    SIPNETParametersV1, InitialConditions, PhotosynthesisParams,
    PhenologyParams, RespirationParams, AllocationParams, WaterParams,
    LeafPhysiologyParams,
)


In [ ]:
BASE_PARAMS = SIPNETParametersV1(
    initial_conditions=InitialConditions(
        plant_wood=30000.0, lai=0.0, soil=10000.0, soil_water_frac=0.5,
        snow=1.0, fine_root_frac=0.05, coarse_root_frac=0.15,
    ),
    photosynthesis=PhotosynthesisParams(
        a_max=120.0, a_max_frac=0.76, base_fol_resp_frac=0.1,
        psn_t_min=2.0, psn_t_opt=20.0, d_vpd_slope=0.05, d_vpd_exp=1.0,
        half_sat_par=200.0, attenuation=0.5,
    ),
    phenology=PhenologyParams(
        leaf_off_day=270.0, gdd_leaf_on=100.0, leaf_growth=50.0,
        frac_leaf_fall=0.95, leaf_allocation=0.25, leaf_turnover_rate=1.0,
    ),
    respiration=RespirationParams(
        base_veg_resp=0.5, veg_resp_q10=2.0, growth_resp_frac=0.0,
        frozen_soil_fol_r_eff=0.5, frozen_soil_threshold=-1.0,
        base_fine_root_resp=0.5, base_coarse_root_resp=0.1,
        fine_root_q10=2.0, coarse_root_q10=2.0,
        base_soil_resp=0.3, soil_resp_q10=2.2, soil_resp_moist_effect=1.5,
    ),
    allocation=AllocationParams(
        fine_root_allocation=0.35, wood_allocation=0.30,
        fine_root_turnover_rate=1.0, coarse_root_turnover_rate=0.1,
        wood_turnover_rate=0.02,
    ),
    water=WaterParams(
        water_remove_frac=0.1, frozen_soil_eff=0.1, wue_const=10.0, soil_whc=12.0,
        litter_whc=5.0, immed_evap_frac=0.1, fast_flow_frac=0.1, snow_melt=0.15,
        rd_const=100.0, r_soil_const1=3.0, r_soil_const2=2.0,
    ),
    leaf=LeafPhysiologyParams(leaf_c_sp_wt=32.0, c_frac_leaf=0.45),
)


In [ ]:
# Climate file is 3-hourly: 8 steps/day
CLIM_PATH = Path("../data/era5_site1.clim")
with open(CLIM_PATH) as f:
    all_lines = f.readlines()

STEPS_PER_DAY = 8
HORIZONS = {
    "1 month":  30 * STEPS_PER_DAY,
    "3 months": 90 * STEPS_PER_DAY,
    "6 months": 180 * STEPS_PER_DAY,
    "1 year":   365 * STEPS_PER_DAY,
    "3 years":  3 * 365 * STEPS_PER_DAY,
}

tmpdir = Path(tempfile.mkdtemp())
slice_paths = {}
for label, n_rows in HORIZONS.items():
    p = tmpdir / f"climate_{label.replace(' ', '_')}.clim"
    p.write_text("".join(all_lines[:n_rows]))
    slice_paths[label] = p

print("Total rows in full file:", len(all_lines))
print("Slice sizes:", {k: v for k, v in HORIZONS.items()})


In [ ]:
# Benchmark: N_REPS runs per horizon
N_REPS = 10
benchmark_results = {}

for label, clim_path in slice_paths.items():
    climate = ClimateDrivers.from_path(clim_path)
    runner = SIPNETRunner(climate_staging=ClimateStaging.SYMLINK)
    model = SIPNETModel(runner, base_params=BASE_PARAMS, base_climate=climate)

    times_ms = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        model()
        times_ms.append((time.perf_counter() - t0) * 1000)

    benchmark_results[label] = (float(np.mean(times_ms)), float(np.std(times_ms)))
    print(f"{label:12s}: {np.mean(times_ms):.1f} +/- {np.std(times_ms):.1f} ms/run")


In [ ]:
labels = list(benchmark_results.keys())
means = [benchmark_results[k][0] for k in labels]
stds  = [benchmark_results[k][1] for k in labels]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(labels)), means, yerr=stds, capsize=5, alpha=0.75, color="steelblue")
ax.axhline(100, color="red", linestyle="--", linewidth=1.5, label="100 ms target")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15)
ax.set_ylabel("Wall time (ms / run)")
ax.set_title("SIPNET single-run latency vs. simulation horizon")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Select and save chosen horizon
chosen_label = None
for label in reversed(labels):
    if benchmark_results[label][0] < 100:
        chosen_label = label
        break

assert chosen_label is not None, "All horizons exceed 100 ms — shorten range"
mean_ms, std_ms = benchmark_results[chosen_label]
print(f"Chosen horizon: {chosen_label}  ({mean_ms:.1f} +/- {std_ms:.1f} ms/run)")

chosen_path = Path("../data/benchmark_climate.clim")
shutil.copy(slice_paths[chosen_label], chosen_path)
print(f"Saved to {chosen_path}  ({HORIZONS[chosen_label]} rows)")

# Verify
c = ClimateDrivers.from_path(chosen_path)
print(f"Timesteps: {c.n_timesteps}, date range: {c.date_range}")

shutil.rmtree(tmpdir)
